# 01 — FineFake download and exploration
MAS Lab 2 — Fake news detection (WCO + MOA)

Goal: get the dataset and understand its structure before deciding how to clean it.

In [1]:
!pip install -q gdown          # "!" runs a shell command; installs the Google Drive downloader quietly (-q)
import os, glob, zipfile, tarfile
import pandas as pd, numpy as np
WORK = '/kaggle/working/finefake'
os.makedirs(WORK, exist_ok=True)

## 1. Download from Google Drive (official link from the SenticNet/FineFake repo)

In [2]:
import gdown
FILE_ID = '16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH'
archive = os.path.join(WORK, 'finefake_archive')
if not os.path.exists(archive):
    gdown.download(id=FILE_ID, output=archive, quiet=False)
print('Size (MB):', round(os.path.getsize(archive)/1e6, 1))
!file {archive}

Size (MB): 5055.9
/kaggle/working/finefake/finefake_archive: Zip archive data, at least v1.0 to extract, compression method=store


## 2. Extraction (zip or tar depending on the format)

In [3]:
if zipfile.is_zipfile(archive):
    with zipfile.ZipFile(archive) as z: z.extractall(WORK)
elif tarfile.is_tarfile(archive):
    with tarfile.open(archive) as t: t.extractall(WORK)
else:
    print('Not an archive: probably the pickle itself')

for root, dirs, files in os.walk(WORK):
    depth = root[len(WORK):].count(os.sep)
    if depth <= 2:
        print('  '*depth + os.path.basename(root) + f'/  ({len(files)} files)')

finefake/  (2 files)
  Image/  (0 files)
    cdc_gov/  (1250 files)
    reddit/  (558 files)
    misleadingthumbnails/  (870 files)
    washingtonpost/  (2095 files)
    nytimes/  (2969 files)
    twitter/  (1020 files)
    oldphotosinreallife/  (348 files)
    photoshopbattles/  (1936 files)
    cnn/  (1413 files)
    snope/  (25880 files)
    historyanecdotes/  (76 files)
    HeresAFunFact/  (410 files)
    apnews/  (1365 files)
    conspiracy/  (453 files)


## 3. Load the DataFrame

In [10]:
pkls = [p for p in glob.glob(WORK + '/**/*', recursive=True) if p.endswith(('.pkl', '.pickle'))]
print(pkls)

src = pkls[0] if pkls else archive

df = pd.read_pickle(src)
DATA_ROOT = os.path.dirname(src)

print(type(df), df.shape)
df.head(3)

['/kaggle/working/finefake/FineFake.pkl']
<class 'pandas.core.frame.DataFrame'> (16909, 13)


,text,image_path,entity_id,topic,fine-grained label,label,knowledge_embedding,description,relation,platform,author,date,comment
0,"CDCs Abortion Surveillance System FAQsNo, stat...",Image/cdc_gov/0.ico,[],Health,0,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[],[],cdc_gov,None,NaN,NaN
1,"Abortion Surveillance — United States, 2012Abo...",Image/cdc_gov/1.ico,"[Q8452, Q113297224, Q3817591, Q30, Q61, Q759, ...",Health,0,1,"[-0.11652757, -0.08209974, -0.06088451, -0.033...","[[Q8452, description, intentional ending of a ...","[[[Q8452, P910, Q6156380], [Q8452, P1343, Q302...",cdc_gov,"[Karen Pazol, Andreea A. Creanga, Denise J. Ja...",NaN,NaN
2,RabiesRabies is a fatal but preventable viral ...,Image/cdc_gov/6.ico,[Q39222],Entertainment,0,1,"[0.034429, -0.098264, 0.168568, -0.169074, 0.0...","[[Q39222, description, deadly viral disease, t...","[[[Q39222, P910, Q7427599], [Q39222, P780, Q10...",cdc_gov,None,NaN,NaN


## 4. Summary (send this output to Claude)

In [5]:
print('Shape:', df.shape)

print('\nColumns / dtypes:'); print(df.dtypes)

print('\nBinary label (0=fake, 1=real):'); print(df['label'].value_counts())

fg = [c for c in df.columns if 'fine' in c.lower()]
if fg: print('\nFine-grained label:'); print(df[fg[0]].value_counts())

print('\nTopics:'); print(df['topic'].value_counts())
print('\nPlatforms:'); print(df['platform'].value_counts())

lens = df['text'].astype(str).str.split().str.len()
print('\nText length (words):'); print(lens.describe().round(1))

print('\nDuplicate texts:', df['text'].duplicated().sum())

Shape: (16909, 13)

Columns / dtypes:
text                   object
image_path             object
entity_id              object
topic                  object
fine-grained label      int64
label                   int64
knowledge_embedding    object
description            object
relation               object
platform               object
author                 object
date                   object
comment                object
dtype: object

Binary label (0=fake, 1=real):
label
0    9407
1    7502
Name: count, dtype: int64

Fine-grained label:
fine-grained label
0    7502
3    3444
4    2574
2    1612
1    1477
5     300
Name: count, dtype: int64

Topics:
topic
Politics         5727
Society          3939
Entertainment    3699
Conflict         1718
Business         1003
Health            710
Uncategorized     113
Name: count, dtype: int64

Platforms:
platform
snope             7556
reddit            4048
cnn               2310
washingtonpost    1041
twitter            952
apnews           

In [6]:
print(df.isna().sum())

img_files = {os.path.basename(p) for p in glob.glob(WORK + '/**/*.*', recursive=True)
             if p.lower().endswith(('.jpg','.jpeg','.png','.gif','.webp','.bmp'))}
print('\nImages found on disk:', len(img_files))

has_img = df['image_path'].astype(str).apply(lambda p: os.path.basename(p) in img_files)
print('Rows with an existing image:', has_img.sum(), '/', len(df))

print('image_path examples:', df['image_path'].head(3).tolist())

text                       0
image_path                 0
entity_id                  0
topic                      0
fine-grained label         0
label                      0
knowledge_embedding        0
description                0
relation                   0
platform                   0
author                  8170
date                    4783
comment                11940
dtype: int64

Images found on disk: 27440
Rows with an existing image: 16763 / 16909
image_path examples: ['Image/cdc_gov/0.ico', 'Image/cdc_gov/1.ico', 'Image/cdc_gov/6.ico']


In [7]:
for lab in [0, 1]:
    print(f'--- label={lab} ---')
    for t in df[df['label']==lab]['text'].astype(str).sample(2, random_state=0):
        print(repr(t[:300])); print()

--- label=0 ---
"Trump has his social media director/ golf caddy tweeting out against fellow Republicans.\n\nIt's only a matter of time people.\n#TrumpRussia https://t.co/UlfNbEPUDo"

'\nThe Canal Saint-Martin in Paris, France, was drained for the first time 200 years, leading to "frightening finds." '

--- label=1 ---
'Anatomy of Detroit’s DeclineAnatomy of Detroit’s Decline\n\nIn a matter of decades, Detroit went from one of America’s most prosperous cities to one of its most distressed. Here is a look at how the collapse of this metropolis – battered by financial missteps, racial tensions and leadership lapses – c'

'\n TIL That no species of centipede has exactly 100 legs\n    '

